[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QuantLet/NN_DL/blob/main/NN_DL_ch02/NN_DL_ch02_VaRForecasting/NN_DL_ch02_VaRForecasting.ipynb)

# NN_DL_ch02_VaRForecasting

**Value-at-Risk Forecasting with a PyTorch MLP**

Download real S&P 500 daily data from stooq.com, engineer financial features (lagged returns, rolling volatility, rolling min/max, squared returns), train an MLP with quantile loss for 1-day 1% VaR forecasting, and compare with Historical Simulation and EWMA VaR. Backtesting via violation rates and the Kupiec likelihood-ratio test.

In [ ]:
!pip install -q torch matplotlib pandas scikit-learn scipy

In [ ]:
# Imports & reproducibility
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import urllib.request
from io import StringIO
from scipy.stats import norm, chi2
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

MAIN_BLUE = "#1A3A6E"
IDA_RED   = "#CD0000"
FOREST    = "#2E7D32"
CRIMSON   = "#DC3545"
AMBER     = "#FFC107"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.grid": True, "grid.alpha": 0.3,
    "font.size": 12, "axes.labelsize": 13, "axes.titlesize": 14,
    "figure.figsize": (10, 6),
})

def save_fig(name):
    plt.savefig(f"{name}.pdf", bbox_inches="tight", dpi=300, transparent=True)
    plt.savefig(f"{name}.png", bbox_inches="tight", dpi=150, transparent=True)
    plt.show()
    print(f"Saved {name}.pdf and {name}.png")

In [ ]:
# Download REAL S&P 500 data from stooq.com
data_source = None
try:
    url = "https://stooq.com/q/d/l/?s=^spx&d1=20200101&d2=20260324&i=d"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=15) as resp:
        raw = resp.read().decode("utf-8")
    df = pd.read_csv(StringIO(raw))
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)
    prices = df["Close"].values.astype(float)
    dates  = df["Date"].values
    data_source = "stooq"
    print(f"Loaded S&P 500 from stooq.com: {len(prices)} daily observations")
except Exception as e1:
    print(f"stooq.com failed ({e1}), trying GitHub mirror...")
    try:
        url2 = "https://raw.githubusercontent.com/datasets/s-and-p-500/main/data/data.csv"
        sp = pd.read_csv(url2)
        date_cols  = [c for c in sp.columns if "date" in c.lower()]
        price_cols = [c for c in sp.columns if any(k in c.lower() for k in ["sp500","price","close","value"])]
        if not price_cols:
            price_cols = [c for c in sp.columns if c not in date_cols]
        sp[date_cols[0]] = pd.to_datetime(sp[date_cols[0]])
        sp = sp.sort_values(date_cols[0]).dropna(subset=[price_cols[0]])
        prices = sp[price_cols[0]].values.astype(float)
        dates  = sp[date_cols[0]].values
        data_source = "github"
        print(f"Loaded S&P 500 from GitHub: {len(prices)} observations")
    except Exception as e2:
        print(f"All downloads failed ({e2}). Generating GARCH(1,1) simulation.")
        np.random.seed(42)
        n = 1500
        omega, alpha, beta = 1e-6, 0.09, 0.90
        sigma2  = np.zeros(n)
        returns_sim = np.zeros(n)
        sigma2[0] = omega / (1 - alpha - beta)
        for t in range(1, n):
            sigma2[t] = omega + alpha * returns_sim[t-1]**2 + beta * sigma2[t-1]
            returns_sim[t] = np.sqrt(sigma2[t]) * np.random.standard_t(df=5)
        prices = 3000 * np.exp(np.cumsum(returns_sim))
        dates  = pd.date_range("2020-01-02", periods=n, freq="B")
        data_source = "simulated"
        print(f"[SIMULATED] Generated {n} GARCH(1,1) prices")

# Compute log-returns
log_returns = np.diff(np.log(prices))
ret_dates   = dates[1:] if len(dates) == len(log_returns) + 1 else np.arange(len(log_returns))
print(f"\nLog-returns: {len(log_returns)} observations")
print(f"Mean={log_returns.mean():.6f}  Std={log_returns.std():.4f}  "
      f"Min={log_returns.min():.4f}  Max={log_returns.max():.4f}")

In [ ]:
# Feature engineering
# Lagged returns (1,2,3,5,10 days), rolling volatility (5,10,20),
# rolling min/max (5,10,20), squared returns
df_feat = pd.DataFrame({"ret": log_returns})

for lag in [1, 2, 3, 5, 10]:
    df_feat[f"ret_lag{lag}"] = df_feat["ret"].shift(lag)

for w in [5, 10, 20]:
    df_feat[f"rvol_{w}"] = df_feat["ret"].rolling(w).std()

for w in [5, 10, 20]:
    df_feat[f"rmin_{w}"] = df_feat["ret"].rolling(w).min()
    df_feat[f"rmax_{w}"] = df_feat["ret"].rolling(w).max()

df_feat["ret_sq"] = df_feat["ret"] ** 2

# Target: next-day return (MLP predicts the 1% quantile)
df_feat["target"] = df_feat["ret"].shift(-1)
df_feat = df_feat.dropna().reset_index(drop=True)

feature_cols = [c for c in df_feat.columns if c not in ["ret", "target"]]
X_all = df_feat[feature_cols].values.astype(np.float32)
y_all = df_feat["target"].values.astype(np.float32)

print(f"Features ({len(feature_cols)}): {feature_cols}")
print(f"Samples after feature engineering: {len(X_all)}")

In [ ]:
# Train/test split (80/20, NO shuffle for time series)
split = int(0.8 * len(X_all))
X_train_raw, X_test_raw = X_all[:split], X_all[split:]
y_train, y_test = y_all[:split], y_all[split:]

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw).astype(np.float32)
X_test  = scaler.transform(X_test_raw).astype(np.float32)

print(f"Train: {X_train.shape}   Test: {X_test.shape}")

In [ ]:
# MLP for VaR (Quantile Regression)
class VaRMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
    def forward(self, x):
        return self.net(x)

model = VaRMLP(X_train.shape[1]).to(device)
print(model)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Quantile loss function
def quantile_loss(pred, target, alpha=0.01):
    diff = target - pred.squeeze()
    return torch.mean(torch.max(alpha * diff, (alpha - 1) * diff))

In [ ]:
# Training loop (100 epochs, Adam, lr=1e-3)
alpha_var = 0.01
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)

val_X = torch.tensor(X_test).to(device)
val_y = torch.tensor(y_test).to(device)

train_losses, val_losses = [], []

for epoch in range(100):
    model.train()
    batch_losses = []
    for xb, yb in train_dl:
        xb, yb = xb.to(device), yb.to(device)
        loss = quantile_loss(model(xb), yb, alpha_var)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
    train_losses.append(np.mean(batch_losses))

    model.eval()
    with torch.no_grad():
        vloss = quantile_loss(model(val_X), val_y, alpha_var).item()
    val_losses.append(vloss)

    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1:3d} | Train QL={train_losses[-1]:.6f}  Val QL={val_losses[-1]:.6f}")

print("Training complete.")

In [ ]:
# Compute VaR forecasts on test set
# 1) Neural-Network VaR
model.eval()
with torch.no_grad():
    var_nn = model(torch.tensor(X_test).to(device)).cpu().numpy().ravel()

# 2) Historical Simulation VaR (rolling 250-day window)
hs_window = 250
returns_full = np.concatenate([y_train, y_test])
var_hs = np.array([
    np.percentile(returns_full[max(0, split+t-hs_window):split+t], alpha_var * 100)
    for t in range(len(y_test))
])

# 3) EWMA VaR (RiskMetrics lambda=0.94)
lam = 0.94
ewma_var2 = np.zeros(len(returns_full))
ewma_var2[0] = returns_full[0] ** 2
for t in range(1, len(returns_full)):
    ewma_var2[t] = lam * ewma_var2[t-1] + (1 - lam) * returns_full[t-1] ** 2
ewma_sigma = np.sqrt(ewma_var2[split:])
var_ewma = norm.ppf(alpha_var) * ewma_sigma

print(f"NN   VaR violations: {(y_test < var_nn).mean():.4f}  (target {alpha_var})")
print(f"HS   VaR violations: {(y_test < var_hs).mean():.4f}  (target {alpha_var})")
print(f"EWMA VaR violations: {(y_test < var_ewma).mean():.4f}  (target {alpha_var})")

In [ ]:
# Backtesting: violation rate + Kupiec LR test
def kupiec_test(violations, n_obs, alpha):
    """Kupiec (1995) Proportion of Failures LR test.
    H0: true violation rate = alpha."""
    x  = int(violations.sum())
    n  = n_obs
    p  = alpha
    ph = x / n if n > 0 else 0
    if ph == 0 or ph == 1:
        return np.nan, np.nan
    lr = -2 * (x * np.log(p / ph) + (n - x) * np.log((1 - p) / (1 - ph)))
    pval = 1 - chi2.cdf(lr, df=1)
    return lr, pval

methods = {
    "Neural Network": var_nn,
    "Hist. Simulation": var_hs,
    "EWMA": var_ewma,
}

header = f"{'Method':<20s} {'Viol.Rate':>10s} {'Expected':>10s} {'Kupiec LR':>10s} {'p-value':>10s}"
print(header)
print("-" * 65)
backtest = {}
for name, var_vec in methods.items():
    viols = y_test < var_vec
    vr = viols.mean()
    lr_stat, pval = kupiec_test(viols, len(y_test), alpha_var)
    backtest[name] = {"viol_rate": vr, "kupiec_lr": lr_stat, "kupiec_p": pval}
    print(f"{name:<20s} {vr:>10.4f} {alpha_var:>10.4f} {lr_stat:>10.3f} {pval:>10.4f}")

In [ ]:
# Plot 1: Returns time series
fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(log_returns, color=MAIN_BLUE, lw=0.5, alpha=0.8)
ax.axhline(0, color="grey", lw=0.5)
ax.axvline(split, color=CRIMSON, ls="--", lw=1.2, label="Train / Test split")
ax.set_xlabel("Observation index")
ax.set_ylabel("Log return")
ax.set_title("S&P 500 Daily Log Returns", fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
save_fig("nn_ch02_returns_timeseries")

In [ ]:
# Plot 2: VaR forecasts with violations highlighted
t = np.arange(len(y_test))
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(t, y_test,   color="grey",    lw=0.5, alpha=0.6, label="Returns")
ax.plot(t, var_nn,   color=MAIN_BLUE, lw=1.5,
        label=f'NN VaR   (viol={backtest["Neural Network"]["viol_rate"]:.3f})')
ax.plot(t, var_hs,   color=IDA_RED,   lw=1.5, ls="--",
        label=f'HS VaR   (viol={backtest["Hist. Simulation"]["viol_rate"]:.3f})')
ax.plot(t, var_ewma, color=FOREST,    lw=1.5, ls=":",
        label=f'EWMA VaR (viol={backtest["EWMA"]["viol_rate"]:.3f})')

viol_mask = y_test < var_nn
ax.scatter(t[viol_mask], y_test[viol_mask], color=CRIMSON, s=18, zorder=5, label="NN Violations")
ax.axhline(0, color="grey", lw=0.5)
ax.set_xlabel("Test period (days)")
ax.set_ylabel("Return")
ax.set_title("1% VaR Forecasts: Neural Network vs Historical Simulation vs EWMA", fontweight="bold")
ax.legend(fontsize=10, loc="lower left")
plt.tight_layout()
save_fig("nn_ch02_var_forecasts")

In [ ]:
# Plot 3: Violation rate bar chart
fig, ax = plt.subplots(figsize=(8, 5))
names  = list(backtest.keys())
rates  = [backtest[n]["viol_rate"] for n in names]
colors = [MAIN_BLUE, IDA_RED, FOREST]

bars = ax.bar(names, rates, color=colors, edgecolor="white", width=0.5)
ax.axhline(alpha_var, color=CRIMSON, ls="--", lw=2, label=f"Expected = {alpha_var}")

for bar, rate in zip(bars, rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f"{rate:.4f}", ha="center", va="bottom", fontweight="bold", fontsize=11)

ax.set_ylabel("Violation Rate")
ax.set_title("VaR Backtesting: Violation Rates (1% level)", fontweight="bold")
ax.legend(fontsize=12)
ax.set_ylim(0, max(rates) * 1.4)
plt.tight_layout()
save_fig("nn_ch02_violation_rates")

In [ ]:
# Plot 4: Training loss curve
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(train_losses, color=MAIN_BLUE, lw=2, label="Train Quantile Loss")
ax.plot(val_losses,   color=IDA_RED,   lw=2, label="Val Quantile Loss")
ax.set_xlabel("Epoch")
ax.set_ylabel("Quantile Loss")
ax.set_title("MLP Training: Quantile Loss Convergence", fontweight="bold")
ax.legend(fontsize=12)
plt.tight_layout()
save_fig("nn_ch02_training_loss")

In [ ]:
# Summary of output files
print("\n=== Output files ===")
for fname in ["nn_ch02_returns_timeseries", "nn_ch02_var_forecasts",
              "nn_ch02_violation_rates", "nn_ch02_training_loss"]:
    print(f"  {fname}.pdf  {fname}.png")
print("\nDone.")

## Summary

Trained a **quantile-regression MLP** (PyTorch) for **1-day 1% Value-at-Risk** forecasting on S&P 500 returns. Features include lagged returns (1-10 days), rolling volatility/min/max (5-20 days), and squared returns. Compared against Historical Simulation and EWMA (RiskMetrics) VaR, and evaluated each method via violation rates and the Kupiec likelihood-ratio test.